To ensure we are not blocked, I have tested this entire pipeline locally and successfully generated our first cleaned dataset (`data_clean_v1.zip`). This V1 dataset was generated by repurposing Ruixuan's logic from the EDA phase.

If Josh finish his better face extraction algorithm, simply replace the logic inside the `get_faces()` function in that file. Once you update `face_utils.py`, everything else will automatically use it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Project/data_raw/deepfake-and-real-images.zip" "/content/"

!unzip -q "/content/deepfake-and-real-images.zip" -d "/content/"

!ls /content/

Dataset  deepfake-and-real-images.zip  drive  sample_data


In [3]:
%%writefile face_utils.py
import cv2
import numpy as np
from typing import List

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_faces(image: np.ndarray) -> List[np.ndarray]:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces_bboxes = face_cascade.detectMultiScale(gray, 1.3, 5)

    face_images = []
    for (x, y, w, h) in faces_bboxes:
        cropped_face = image[y:y+h, x:x+w]
        face_images.append(cropped_face)

    return face_images

Writing face_utils.py


In [8]:
%%writefile clean_data.py
import os
import cv2
import shutil
from tqdm import tqdm
from face_utils import get_faces

def clean_and_prepare_data(raw_dir="/content/Dataset", clean_dir="/content/data_clean_v1"):
    if os.path.exists(clean_dir):
        shutil.rmtree(clean_dir)
    os.makedirs(clean_dir)

    splits = ['Train', 'Validation', 'Test']
    categories = ['Real', 'Fake']

    for split in splits:
        for category in categories:
            raw_path = os.path.join(raw_dir, split, category)
            clean_path = os.path.join(clean_dir, split, category)
            os.makedirs(clean_path, exist_ok=True)

            if not os.path.exists(raw_path):
                continue

            images = os.listdir(raw_path)

            for img_name in tqdm(images, desc=f"{split}/{category}"):
                img_path = os.path.join(raw_path, img_name)

                if split in ['Test', 'Validation']:
                    shutil.copy2(img_path, os.path.join(clean_path, img_name))
                    continue

                img = cv2.imread(img_path)
                if img is None: continue

                faces = get_faces(img)
                if len(faces) == 0: continue

                for i, face_img in enumerate(faces):
                    name, ext = os.path.splitext(img_name)
                    new_img_name = f"{name}_face{i}{ext}"
                    new_img_path = os.path.join(clean_path, new_img_name)
                    cv2.imwrite(new_img_path, face_img)

    print(f"Zipping the cleaned dataset...")
    shutil.make_archive('/content/data_clean_v1', 'zip', clean_dir)
    print("Done!")

if __name__ == "__main__":
    clean_and_prepare_data()

Overwriting clean_data.py


In [6]:
!python clean_data.py

Train/Real: 100% 70001/70001 [24:35<00:00, 47.45it/s]
Train/Fake: 100% 70001/70001 [24:51<00:00, 46.94it/s]
Validation/Real: 100% 19787/19787 [00:03<00:00, 5472.71it/s]
Validation/Fake: 100% 19641/19641 [00:04<00:00, 4683.81it/s]
Test/Real: 100% 5413/5413 [00:02<00:00, 1827.34it/s]
Test/Fake: 100% 5492/5492 [00:03<00:00, 1564.20it/s]
Zipping the cleaned dataset...
Done!


In [7]:
!cp "/content/data_clean_v1.zip" "/content/drive/MyDrive/Project/data_clean/"

In [ ]:
%%writefile inference.py
import torch
from torchvision.transforms import v2 as T
from PIL import Image
import cv2

from face_utils import get_faces

# Note: The model class (DeppFakeDetector) should be passed as an argument when calling is_deepfake() from another script/notebook.

def is_deepfake(image_path: str, model: torch.nn.Module, device: torch.device, transform) -> tuple:
    """
    Determines if an image is a Deepfake.
    Logic: Extracts all faces. If ANY face is classified as Fake (<0), the image is a Deepfake.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image at {image_path}")

    faces = get_faces(img)

    if len(faces) == 0:
        return False, "No face detected in the image."

    model.eval()

    with torch.no_grad():
        for face_array in faces:
            face_rgb = cv2.cvtColor(face_array, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(face_rgb)

            input_tensor = transform(pil_image).unsqueeze(0).to(device)

            output = model(input_tensor)
            if output.item() < 0:
                return True, "Deepfake detected!"

    return False, "All faces appear to be real."